#Preparar datos sobre Naves Imperiales de Star Wars (v2)


In [ ]:
#@title Librerías

import sys, os, re, random
import numpy as np
import pandas as pd
from google.colab import files


import os
import csv

import ipywidgets as widgets
from ipywidgets import Box, Layout
from IPython.display import clear_output
import random

print("Librerías cargadas.")

Librerías cargadas.


#Cargar datos descargados

In [ ]:
# @title Acceder al Drive {"single-column":true}

# Nota: la primera vez se debe confirmar el uso logueandose en "Google Drive File Stream" y obteniendo código de autentificación.
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

# directorio local en Google Drive
path = '/content/gdrive/MyDrive/demosColab/demoStarWars/datos/'  #@param {type:"string"}


Mounted at /content/gdrive


In [ ]:
#@title Cargar datos

#@markdown ### Archivo de datos a utilizar:
archivo_datos = 'navesOri.csv'  #@param {type:"string"}
#@markdown ### Configuración del archivo CSV:
delimitador_columnas = ',' #@param {type:"string"}
separador_decimal = '.' #@param {type:"string"}

# funciones auxiliares

# importa datos
def cargarDatosDF(path, archivo_datos, delimitador_columnas, separador_decimal, mostrarEstadisticas=True):
  if os.path.isfile( path + '/' + archivo_datos ):
    # existe el archivo
    if ((delimitador_columnas is None) or (delimitador_columnas=="")):
      # si no se define asume ","
      delimitador_columnas = ","
    if ((separador_decimal is None) or (separador_decimal=="")):
      # si no se define asume "."
      separador_decimal = "."
    if (delimitador_columnas==separador_decimal):
      # ambos delimitadores iguales, cambia el decimal
      if delimitador_columnas == ",":
        separador_decimal = "."
      else:
        separador_decimal = ","
      print("- Ambos delimitadores configurados igual, se cambia separador decimal a '" + separador_decimal + "'!")
    # carga datos
    df = pd.read_csv(path + archivo_datos,
                     sep=delimitador_columnas, decimal=separador_decimal,
                     skip_blank_lines=True,
                     engine="python")
    print("> Archivo de datos", archivo_datos, "cargado")
    # muestra estadísticas
    if mostrarEstadisticas:
      print("\n> Cabecera: ")
      print(df.head())
      print("\n> Características: ")
      print(df.describe())
      print("\n")
    # controla que el archivo tenga sentido
    if len(df.columns.values.tolist())<2:
      print("\n> El archivo de datos debería tener al menos 2 columnas, revise delimitador de columnas!")
      return None
    else:
      return df
  else:
    print("No existe archivo de datos ", archivo_datos, "!")
    return None

# importa definición axiliar de las clases (si existe)
def cargarNombreClases(path, archivo_datos):
  # importa definición de la clase
  arClasesFN = archivo_datos.split('.')[0] + '_nombreClases.txt'
  if os.path.isfile( path + '/' + arClasesFN ):
    # si existe, carga los datos
    with open( path + '/' + arClasesFN, mode='r') as csvfile:
        r = csv.reader(csvfile, delimiter=',')
        auxAtributo = r.__next__()
        auxClases = r.__next__()
    print('\n> Definición de los valores discretos para la clase cargada de ' + arClasesFN +'.\n')
    return auxAtributo[0], ','.join(auxClases)
  else:
    # no encontrado
    return "", ""


# configura para que muestre todas las columnas y filas
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

## aplicación de los parámetros elegidos

# Carga los datos del CSV y muestra los primeros
df = cargarDatosDF(path, archivo_datos,
                   delimitador_columnas, separador_decimal, mostrarEstadisticas=True)



> Archivo de datos navesOri.csv cargado

> Cabecera: 
                         Name           Ship Type  \
0             Builder Shuttle       Landing Craft   
1                      TIE X3        TIE Fighters   
2  Star Galleon Class Frigate        Medium Ships   
3               A-9 Vigilance  Other Starfighters   
4        TIE Ground Targeting         TIE Bombers   

                               Model          Manufacturer      Length Crew  \
0  Builder Shuttle Mark 1 and Mark 2     Cygnus Spaceworks   40 meters    4   
1                     TIE/x3 Fighter  Sienar Fleet Systems  7.8 meters    1   
2         Star Galleon Class Frigate      Kuat Drive Yards  298 meters  150   
3                      A-9 Vigilance      Kuat Drive Yards  7.4 meters    1   
4                     TIE/gt Fighter  Sienar Fleet Systems  6.3 meters    1   

                                     Cargo Capacity Consumables  \
0  15 metric tons (Mark 2 can hold 400 metric tons)      1 week   
1                 

In [ ]:
#@title Mostrar Estadísticas de datos recolectados


# variables auxiliares
atributo_clase = ""

# configura para que muestre todas las columnas y filas
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

# devuelve listas de columnas numéricas y no numéricas
def devolNombreColumnas(ndf):
  colValues = []
  colNoValues = []
  for col in ndf.columns:
    if ndf[col].dtypes in ("object", "bool"):
      colNoValues.append( col )
    else:
      colValues.append( col )
  return colValues, colNoValues

# función auxiliar para separar datos de entrada y de salida
def separarDatosXY(ndf, atributo_clase="", xSoloNros=True):
  # hace una copia auxiliar del data frame
  cdf = ndf.copy()
  # saca el atributo clase (OPCIONAL)
  if atributo_clase == "":
    Y = []
  else:
    # datos atributo clase
    Y = np.array( cdf.pop(atributo_clase).fillna("-NAN-") )
  if xSoloNros:
    # se queda sólo con los atributos numéricos (OPCIONAL)
    for col in cdf.columns:
      if cdf[col].dtypes == "object":
          cdf.pop( col )
  # datos de entrada
  X = np.array(cdf.infer_objects(copy=False).fillna(0.001))
  return X, Y, np.array(cdf.columns)

def convColsNumericas(ndf, atributos_no_convertir = []):
  # hace una copia auxiliar del data frame
  cdf = ndf.copy()
  # convierte todas las no numéricas a numéricas (OPCIONAL)
  for col in cdf.columns:
    if col not in atributos_no_convertir:
      if cdf[col].dtypes == "object":
        # genera diccionario de valores
        valores = cdf[col].unique()
        diccValores = dict(zip(valores, range(len(valores))))
        # realiza el reemplazo
        cdf[col] = cdf[col].map(lambda s: diccValores.get(s) if s in diccValores else s)
  return cdf

# función auxiliar
def generar_estadisticas_detalladas(orDF, titulo=""):
  # título
  print("\n", titulo, ": ")
  # obtiene las estadísticas generales
  estDF = orDF.describe().transpose()
  #  genera y formatea las estadísticas
  if "min" in estDF and "max" in estDF:
    rangoValores = "[ " + estDF["min"].apply('{:.2f}'.format) + " ; " + estDF["max"].apply('{:.2f}'.format) + " ]"
  else:
    rangoValores = estDF["unique"].infer_objects(copy=False).fillna(0.0).apply('{:.0f}'.format)
  rangoValores.name = "Rango Valores"
  # para campos no numéricos muestra las cantidades por valor
  for col in orDF.columns:
    if orDF[col].dtypes in ("object", "bool"):
      auxStr = str( orDF[col].value_counts() ).replace("\n", " ; ")
      if (auxStr.index("Name")-3) > 0:
        # saca lo del final porque no sirve
        auxStr = auxStr[:auxStr.index("Name")-3]
      rangoValores[col] = "{ " + auxStr + " }"
  if "mean" in estDF and "std" in estDF:
    promValores = estDF["mean"].fillna(0.0).apply('{:.3f}'.format) + " ± " + estDF["std"].fillna(0.0).apply('{:.3f}'.format)
  else:
    promValores = estDF["count"].apply('{:.0f}'.format)
  promValores.name = "Promedio ± Desvío"
  # obtiene valores "ceros" y nulos
  zero_val = (orDF == 0.00).astype(int).sum(axis=0)
  zero_val.name = "¿Valores Ceros?"
  mis_val = orDF.isnull().sum()
  mis_val.name = "¿Valores Nulos?"
  # prepara la nueva tabla para mostrar
  nTable = pd.concat([orDF.dtypes, rangoValores, promValores, zero_val, mis_val], axis=1)
  nTable = nTable.rename( columns = {0: 'Tipo Valor',  1: 'Rango Valores', 2: 'Promedio ± Desvío', 3: '¿Valores Ceros?', 4: '¿Valores Nulos?' } )
  # muestra la nueva tabla
  pd.set_option('max_colwidth', None)
  display(nTable.fillna("-"))
  print("Tiene " + str(orDF.shape[1]) + " atributos y " + str(orDF.shape[0]) + " ejemplos.")
  print("\n")
  return

# muestra las estadísticas
generar_estadisticas_detalladas(df, "> Estadísticas de los datos recolectados")



 > Estadísticas de los datos recolectados : 


Tipo Valor  \
Name                      object   
Ship Type                 object   
Model                     object   
Manufacturer              object   
Length                    object   
Crew                      object   
Cargo Capacity            object   
Consumables               object   
Hyperdrive Multiplier     object   
Hyperdrive Backup         object   
Speed                     object   
Hull                      object   
Special Features          object   
URL_data                  object   
URL_image                 object   
Shields                   object   
Weapons                   object   
Troops                    object   
Onboard Craft             object   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

Tiene 19 atributos y 107 ejemplos.




#Preparar Datos de Naves Imperiales

In [ ]:
#@title Hacer copia de trabajo (para no romper datos cargados)
ndf = df.copy()

ndf.head()

,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,Cygnus Spaceworks,40 meters,4,15 metric tons (Mark 2 can hold 400 metric tons),1 week,none (Mark 2 has x1),none (Mark 2 has x10),35 MGLT,45 RU,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,NaN,NaN,NaN,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,Sienar Fleet Systems,7.8 meters,1,150 kilograms,5 days,NaN,NaN,110 MGLT,14 RU,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24 SBD,2 Laser Cannons,NaN,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,Kuat Drive Yards,298 meters,150,"100,000 metric tons",6 months,x2,x15,18 MGLT,228 RU,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320 SBD,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,Kuat Drive Yards,7.4 meters,1,55 kilograms,1 day,NaN,NaN,115 MGLT,16 RU,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,NaN,2 Laser Cannons,NaN,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,Sienar Fleet Systems,6.3 meters,1,65 kilograms,2 days,NaN,NaN,75 MGLT,9 RU,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,NaN,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",NaN,NaN


## Desglosar Ejemplos

In [ ]:
#@title Desglogar Mu-1, Mu-2 and Mu-3 Class Shuttle


# parámetros a considerar
modelName = "Mu Class Shuttle"
listaModelsShuttle  = [ "Mu-1", "Mu-2", "Mu-3" ]

# obtiene ejemplo de la nave a corregir
auxDF = ndf[ ndf["Name"] == modelName]
rowShuttle = list(auxDF.values)

print("\n Ejemplo original: ")
display(auxDF)
print("")


# si la encunetra
if (rowShuttle is None) or (len(rowShuttle)<1) or (len(rowShuttle[0])<1):
  print("No se encuentra ejemplo de " + modelName +"!")
else:

  # obtiene valores
  rowShuttle = rowShuttle[0]

  # elimina ejemplo del dataframe
  ndf = ndf.drop(auxDF.index[0])

  # procesa por tipos de modelos
  rangeCantModels = range(len(listaModelsShuttle))
  newRows = []
  for i in rangeCantModels:
    newR = []
    modelShuttle = listaModelsShuttle[i]
    # procesa valores del ejemplo
    for val in rowShuttle:
      # si figura modelo (sino no hace nada)
      if modelShuttle in str(val):
        # obtiene posicion model
        posModelShuttle = val.find(modelShuttle)
        posSeparador = val.find("/")
        if posSeparador > 0:
          # si está separador
          if posModelShuttle < posSeparador:
            # elimina texto anterior al separador
            val = val[:posSeparador]
          else:
            # elimina texto posterior al separador
            val = val[posSeparador+1:]
        # limpia para extraer valor correspondiente
        for j in rangeCantModels:
          if (posSeparador > 0) or (i!=j):
            val = val.replace(listaModelsShuttle[j], "")
        val = val.replace(" and ", "")
        val = val.replace(", ", "")
        val = val.replace(":", "")
        val = val.strip()
      # agrega
      newR.append( val )
    # agrega nuevo ejemplo
    newRows.append( newR )

  # agrega nuevos ejemplos al dataframe
  newDF = pd.DataFrame(newRows, columns=ndf.columns)
  ndf = pd.concat([ndf, newDF], ignore_index=True)

  # muestra nuevos
  print("\n\n Ejemplos desglosados: ")
  display( ndf[ ndf["Name"] == modelName] )
  print("")


 Ejemplo original: 


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
16,Mu Class Shuttle,Shuttles,"Mu-1, Mu-2 and Mu-3 Class Shuttle",Cygnus Spaceworks / Sienar Fleet Systems,20 meters,2,Mu-1 and Mu-2: 100 metric tons / Mu-3: 50 metric tons,Mu-1 and Mu-2: 6 months / Mu-3: 2 months,x2,x20,67 MGLT,25 RU,Adjustable stabilizer fins.,http://insd.swcombine.com/insd/mu.htm,http://insd.swcombine.com/insd/Ship_files/mu.jpg,Mu-1 and Mu-2: 100 SBD / Mu-3: 120 SBD,Mu-1 and Mu-2: 2 Laser Cannons / Mu-3: 2 Medium Laser Cannons.,Mu-1: 24 / Mu-2: 14 / Mu-3: 40,NaN





 Ejemplos desglosados: 


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
106,Mu Class Shuttle,Shuttles,Mu-1 Class Shuttle,Cygnus Spaceworks / Sienar Fleet Systems,20 meters,2,100 metric tons,6 months,x2,x20,67 MGLT,25 RU,Adjustable stabilizer fins.,http://insd.swcombine.com/insd/mu.htm,http://insd.swcombine.com/insd/Ship_files/mu.jpg,100 SBD,2 Laser Cannons,24,NaN
107,Mu Class Shuttle,Shuttles,Mu-2 Class Shuttle,Cygnus Spaceworks / Sienar Fleet Systems,20 meters,2,100 metric tons,6 months,x2,x20,67 MGLT,25 RU,Adjustable stabilizer fins.,http://insd.swcombine.com/insd/mu.htm,http://insd.swcombine.com/insd/Ship_files/mu.jpg,100 SBD,2 Laser Cannons,14 / 40,NaN
108,Mu Class Shuttle,Shuttles,Mu-3 Class Shuttle,Cygnus Spaceworks / Sienar Fleet Systems,20 meters,2,50 metric tons,2 months,x2,x20,67 MGLT,25 RU,Adjustable stabilizer fins.,http://insd.swcombine.com/insd/mu.htm,http://insd.swcombine.com/insd/Ship_files/mu.jpg,120 SBD,2 Medium Laser Cannons.,14 / 40,NaN


## Ajustar Formato de Campos

In [ ]:
#@title Ajustar campos Length, Hyperdrive Multiplier, Speed, Hull, Shields  donde se extrae parte numérica eliminando texto

# funciones auxiliares
def extraerPrimerNro(columnName):
  global ndf
  # siempre elimina "," como separador de miles
  reemplazar(columnName, ",")
  print("\t extrae 1er número")
  # toma número con decimales
  ndf[columnName] = ndf[columnName].astype(str).str.extract(r'(\d+\.?\d*)\D*', expand=True)
  return

def corregirRangos(columnName):
  global ndf
  # obtiene valores que son rangos
  searchMask = ndf[columnName].astype(str).str.contains('-')
  if len(ndf.loc[searchMask, columnName])>0:
    # procesa valores
    newVals = []
    for val in ndf.loc[searchMask, columnName]:
      nval = val
      #controla que sea rango válido
      if "-" in val:
        posSep1 = val.index("-")
        if posSep1>0:
          if " " in val:
            posSep2 = val.index(" ")
          else:
            posSep2 = len(val)
          # calcula promedio
          nval = ( float(val[0:posSep1].replace(",", "")) + float(val[posSep1+1:posSep2].replace(",", "")) ) / 2
          nval = str(nval) + val[posSep2:]
          print("\t rango: ", val, "--> asigna = ", nval)
      newVals.append(nval)
    # reemplaza valores
    ##print("\n ant: ", ndf.loc[searchMask, columnName])
    ndf.loc[searchMask, columnName] = newVals
    ##print("\n new: ", ndf.loc[searchMask, columnName])

def reemplazarRegEx(columnName, oldValue, newValue=""):
  global ndf
  #print("\t reemplaza regex '" + str(oldValue) + "' por '" + str(newValue) + "'")
  ndf[columnName] = ndf[columnName].astype(str).str.replace(oldValue, newValue, regex=True)
  ndf[columnName] = ndf[columnName].fillna(0)
  return

def reemplazar(columnName, oldValue, newValue=""):
  global ndf
  print("\t reemplaza '" + str(oldValue) + "' por '" + str(newValue) + "'")
  ndf[columnName] = ndf[columnName].astype(str).str.replace(oldValue, newValue, regex=False)
  ndf[columnName] = ndf[columnName].fillna(0)
  return

def eliminarDatosEntreParentesis(columnName):
  print("\t elimina texto entre paréntesis")
  reemplazarRegEx(columnName, r'\s*\(.*?\)', "")
  return

def completarVacios(columnName, newValue=-1):
  global ndf
  print("\t completar vacíos con " + str(newValue) + "")
  ndf[columnName] = ndf[columnName].fillna(newValue)
  return

def convertirFloat(columnName):
  global ndf
  print("\t convierte a float")
  ndf[columnName] = ndf[columnName].astype(float)

print("\n-procesa Length")
reemplazar("Length", " meters")
eliminarDatosEntreParentesis("Length")
corregirRangos("Length")
extraerPrimerNro("Length")
completarVacios("Length", 0)
convertirFloat("Length")

print("\n-procesa Hyperdrive Multiplier")
eliminarDatosEntreParentesis("Hyperdrive Multiplier")
reemplazar("Hyperdrive Multiplier", "x")
reemplazar("Hyperdrive Multiplier", ",", ".")
eliminarDatosEntreParentesis("Hyperdrive Multiplier")
corregirRangos("Hyperdrive Multiplier")
extraerPrimerNro("Hyperdrive Multiplier")
completarVacios("Hyperdrive Multiplier", -1)
convertirFloat("Hyperdrive Multiplier")

print("\n-procesa Speed")
reemplazar("Speed", " MGLT")
eliminarDatosEntreParentesis("Speed")
corregirRangos("Speed")
extraerPrimerNro("Speed")
completarVacios("Speed", 0)
convertirFloat("Speed")

print("\n-procesa Hull")
reemplazar("Hull", " RU")
eliminarDatosEntreParentesis("Hull")
corregirRangos("Hull")
extraerPrimerNro("Hull")
completarVacios("Hull", 0)
convertirFloat("Hull")

print("\n-procesa Shields")
reemplazar("Shields", " SBD")
eliminarDatosEntreParentesis("Shields")
corregirRangos("Shields")
extraerPrimerNro("Shields")
completarVacios("Shields", -1)
convertirFloat("Shields")

# muestra cambios
ndf.head()


-procesa Length
	 reemplaza ' meters' por ''
	 elimina texto entre paréntesis
	 rango:  2,500-17,600 --> asigna =  10050.0
	 reemplaza ',' por ''
	 extrae 1er número
	 completar vacíos con 0
	 convierte a float

-procesa Hyperdrive Multiplier
	 elimina texto entre paréntesis
	 reemplaza 'x' por ''
	 reemplaza ',' por '.'
	 elimina texto entre paréntesis
	 rango:  3-1 --> asigna =  2.0
	 reemplaza ',' por ''
	 extrae 1er número
	 completar vacíos con -1
	 convierte a float

-procesa Speed
	 reemplaza ' MGLT' por ''
	 elimina texto entre paréntesis
	 rango:  8-4 --> asigna =  6.0
	 reemplaza ',' por ''
	 extrae 1er número
	 completar vacíos con 0
	 convierte a float

-procesa Hull
	 reemplaza ' RU' por ''
	 elimina texto entre paréntesis
	 rango:  2,840-45,712 --> asigna =  24276.0
	 reemplaza ',' por ''
	 extrae 1er número
	 completar vacíos con 0
	 convierte a float

-procesa Shields
	 reemplaza ' SBD' por ''
	 elimina texto entre paréntesis
	 rango:  6,000-96,000 --> asigna =  51000.

,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,Cygnus Spaceworks,40.0,4,15 metric tons (Mark 2 can hold 400 metric tons),1 week,-1.0,none (Mark 2 has x10),35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,NaN,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,Sienar Fleet Systems,7.8,1,150 kilograms,5 days,-1.0,NaN,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,NaN,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,Kuat Drive Yards,298.0,150,"100,000 metric tons",6 months,2.0,x15,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,Kuat Drive Yards,7.4,1,55 kilograms,1 day,-1.0,NaN,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,NaN,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,Sienar Fleet Systems,6.3,1,65 kilograms,2 days,-1.0,NaN,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",NaN,NaN


In [ ]:
#@title Preparar campo Manufacturer

print("-procesa Manufacturer (asigna codigo valor ID)")

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder().fit(ndf["Manufacturer"])
ndf["Manufacturer"] = encoder.transform(ndf["Manufacturer"])
convertirFloat("Manufacturer")

# muestra códificación asignada
print("\n\tCodificación asignada: ")
for i in range(len(encoder.classes_)):
  print("\t\t", i, ":", encoder.classes_[i])
print("")

ndf.head()

-procesa Manufacturer (asigna codigo valor ID)
	 convierte a float

	Codificación asignada: 
		 0 : Byss Worx / Imperial Department of Military Research
		 1 : CPG Space Products
		 2 : Corellian Engineering Corporation
		 3 : Cygnus Spaceworks
		 4 : Cygnus Spaceworks / Sienar Fleet Systems
		 5 : Damorian Manufacturing Corporation
		 6 : Imperial Shipyards
		 7 : Incom Corporation
		 8 : Kuat Drive Yards
		 9 : Loronar
		 10 : Loronar / Rendili StarDrive /Sienar Fleet Systems and Kuat Drive Yards
		 11 : Meller & Dax
		 12 : Mesens Corporation
		 13 : Rendili StarDrive
		 14 : Republic Sienar Systems
		 15 : Rothana Heavy Engineering
		 16 : Rothana Heavy Engineering / Kuat Drive Yards
		 17 : Santhe / Sienar Technologies
		 18 : Sienar Fleet Systems
		 19 : Sienar Fleet Systems / Chiss Ascendancy
		 20 : Sienar Fleet Systems / Imperial Department of Military Research
		 21 : Sienar Fleet Systems / Shobquix Yards
		 22 : Sienar Fleet Systems / Zsinj Development Incorporated
		 23 : S

,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4,15 metric tons (Mark 2 can hold 400 metric tons),1 week,-1.0,none (Mark 2 has x10),35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,NaN,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1,150 kilograms,5 days,-1.0,NaN,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,NaN,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150,"100,000 metric tons",6 months,2.0,x15,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1,55 kilograms,1 day,-1.0,NaN,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,NaN,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1,65 kilograms,2 days,-1.0,NaN,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",NaN,NaN


In [ ]:
#@title Preparar campo Cargo Capacity

print("-procesa Cargo Capacity (pasa todo a kilogramos)")

# realiza los cambios considerando tipo de métrica
auxOriList = list(ndf["Cargo Capacity"])
auxNewList = []
for val in auxOriList:
  #print(val)
  if (val is None) or (str(val) in ["nan", "(varies according to mission profile)"]):
    auxNewList.append( -1 )
  else:
    val = val.replace(",", "")
    val = val.replace(" + 1", "")
    if "(" in val:
      posParent = val.index("(")
    else:
      posParent = -1
    if "-" in val:
      # calcula promedio
      posSep1 = val.index("-")
      if " " in val:
        posSep2 = val.index(" ")
      else:
        posSep2 = len(val)
      nval = ( float(val[0:posSep1]) + float(val[posSep1+1:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "or" in val:
      # calcula promedio
      posSep1 = val.index(" or ")
      posSep2 = val[posSep1+4:].index(" ") + posSep1 + 4
      nval = ( float(val[0:posSep1]) + float(val[posSep1+4:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "kilograms" in val:
      posAux = val.index("kilograms")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) )
    elif "metric ton" in val:
      posAux = val.index("metric ton")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) * 1000 )
    else:
      print("No se puede procesar: ", val)

# actualiza los datos
ndf["Cargo Capacity"] = auxNewList
convertirFloat("Cargo Capacity")


ndf.head()

-procesa Cargo Capacity (pasa todo a kilogramos)
	 convierte a float


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4,15000.0,1 week,-1.0,none (Mark 2 has x10),35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,NaN,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1,150.0,5 days,-1.0,NaN,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,NaN,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150,100000000.0,6 months,2.0,x15,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1,55.0,1 day,-1.0,NaN,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,NaN,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1,65.0,2 days,-1.0,NaN,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",NaN,NaN


In [ ]:
#@title Preparar campo Weapons

print("-procesa Weapons (extrae lista de armas y genera nuevas columnas numéricas)")

import re

newValues = []

for f in ndf["Weapons"]:
  # extrae la lista de armas
  auxSplit =  re.split(", | and | AND | or | OR ", str(f))
  ##print(auxSplit)
  # define columnas en base subcadenas obtenidas
  auxDict = {}
  for w in auxSplit:
    if w=="nan":
      continue
    res = re.search(r'(\d+\.?\d*)\s(.+)', w)
    if (res is not None):
      ##print(res.group(1), "*", res.group(2))
      auxW = str(res.group(2)).replace(".", "").upper().strip()
      if "(" in auxW:
        # saca toda aclaración entre paréntesis
        auxW = auxW[:auxW.find("(")].strip()
      # si no es plural, le agrega S (para unificar columnas)
      if auxW[len(auxW)-1] != "S":
        auxW = auxW + "S"
      auxQ = float(res.group(1))
    else:
      auxW = w.upper().strip()
      auxQ = 1.0
    if len(auxW)<1:
      auxW = "OTHER"
    if auxW in auxDict:
      auxDict[auxW] = auxDict[auxW] + auxQ
    else:
      auxDict[auxW] = auxQ
  newValues.append( auxDict )


print("")

# consolida nombre de las armas usando listas predefindas de manera que se reduzca la cantidad de atributios
weaponTypeDict = {"GRAVITY WELL":"GRAVITY WELL",
                  "TRACTORBEAM":"TRACTOR BEAM",
                  "TRACTOR BEAM": "TRACTOR BEAM",
                  "SUPERLASERS":"SUPERLASERS",
                  "TORPEDO":"TORPEDO",
                  "BLASTER":"BLASTER",
                  "ION":"ION",
                  "EMP":"EMP",
                  "TURBOLASER":"TURBOLASER",
                  "LASER":"LASER",
                  "SUPERLASER":"SUPERLASER",
                  "MISSILE":"MISSILE",
                  "THERMAL DETONATOR":"THERMAL DETONATOR",
                  "HULL-CUTTING AIRLOCKS":"HULL-CUTTING AIRLOCKS",
                  "ORBITAL MINE":"ORBITAL MINE",
                  "SPACE BOMB":"SPACE BOMB",
                  "GENERAL PURPOSE":"GENERAL PURPOSE"}

weaponSufixDict = {"BATTER":"BATTERIES",
                  "TURRET":"TURRETS",
                  "CANNON":"CANNONS",
                  "LAUNCHER":"LAUNCHER",
                  "TUBE":"TUBES",
                  "PROJECTOR":"PROJECTORS" }

weaponPrefixDict = {"DOUBLE":2,
                  "TWIN":2,
                  "HEAVY":10,
                  "MEDIUM":5,
                  "LIGHT":1,
                  "QUAD":4 }


weaponTypeDict_keys = list(weaponTypeDict.keys())
weaponSufixDict_keys = list(weaponSufixDict)
weaponPrefixDict_keys = list(weaponPrefixDict)


if len(weaponTypeDict_keys)>0:
  newValuesConsol = []
  # recorre valores separados
  # y si encuentra en lista consolidada lo registra con ese nombre
  for t in newValues:
      nt = {}
      for k in t.keys():
          newK = None
          suffVal = 1
          # busca por weapon type
          for n in weaponTypeDict_keys:
            if n in k:
              newK = weaponTypeDict[n]
              break
          if newK is None:
            print("*", k, "no asociado con weapon type!")
          else:
            # busca por weapon suffix (opcional?)
            sk = None
            for n in weaponSufixDict_keys:
              if n in k:
                sk =  weaponSufixDict[n]
                break
            if sk is not None:
              newK = newK + " " + sk
            # busca por weapon prefix
            for n in weaponPrefixDict_keys:
              if n in k:
                suffVal = weaponPrefixDict[n]
                break
            # registra
            if newK in nt:
              nt[newK] += t[k]*suffVal
            else:
              nt[newK] = t[k]*suffVal
      newValuesConsol.append( nt )
  newValues = newValuesConsol

print("")

# crea nuevo data frame
auxDF = pd.DataFrame.from_dict(newValues)
# completa valores vacios con cero
for col in auxDF:
  auxDF[col] = auxDF[col].fillna(0)
# agrega al data frame anterior y elimina columna original
ndf = pd.concat([ndf, auxDF], axis=1)
ndf = ndf.drop(["Weapons"], axis=1)

ndf.head()


-procesa Weapons (extrae lista de armas y genera nuevas columnas numéricas)

* (VARIES ACCORDING TO SHIP MODEL) no asociado con weapon type!



,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,ION LAUNCHER,GENERAL PURPOSE LAUNCHER,TURBOLASER TURRETS,BLASTER CANNONS,LASER TURRETS,TRACTOR BEAM PROJECTORS,TURBOLASER BATTERIES,ION CANNONS,TORPEDO LAUNCHER,GRAVITY WELL PROJECTORS,SUPERLASERS,BLASTER TURRETS,TORPEDO,ORBITAL MINE,THERMAL DETONATOR LAUNCHER,TURBOLASER,ION TURRETS,SPACE BOMB,ION BATTERIES,HULL-CUTTING AIRLOCKS,EMP CANNONS,TORPEDO TUBES
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4,15000.0,1 week,-1.0,none (Mark 2 has x10),35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1,150.0,5 days,-1.0,NaN,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,NaN,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150,100000000.0,6 months,2.0,x15,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,300,NaN,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1,55.0,1 day,-1.0,NaN,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,NaN,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1,65.0,2 days,-1.0,NaN,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,NaN,NaN,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#@title Preparar campos Onboard Craft & Special Features como binarios (tiene/no tiene)

def transformaBinario(columnName):
  global ndf
  print("-procesa " + columnName)
  ndf[columnName] = np.where(ndf[columnName].isna(), "0", "1")
  convertirFloat(columnName)

transformaBinario("Onboard Craft")
transformaBinario("Special Features")

ndf.head()

-procesa Onboard Craft
	 convierte a float
-procesa Special Features
	 convierte a float


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,ION LAUNCHER,GENERAL PURPOSE LAUNCHER,TURBOLASER TURRETS,BLASTER CANNONS,LASER TURRETS,TRACTOR BEAM PROJECTORS,TURBOLASER BATTERIES,ION CANNONS,TORPEDO LAUNCHER,GRAVITY WELL PROJECTORS,SUPERLASERS,BLASTER TURRETS,TORPEDO,ORBITAL MINE,THERMAL DETONATOR LAUNCHER,TURBOLASER,ION TURRETS,SPACE BOMB,ION BATTERIES,HULL-CUTTING AIRLOCKS,EMP CANNONS,TORPEDO TUBES
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4,15000.0,1 week,-1.0,none (Mark 2 has x10),35.0,45.0,1.0,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1,150.0,5 days,-1.0,NaN,110.0,14.0,0.0,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150,100000000.0,6 months,2.0,x15,18.0,228.0,1.0,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,300,0.0,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1,55.0,1 day,-1.0,NaN,115.0,16.0,0.0,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1,65.0,2 days,-1.0,NaN,75.0,9.0,0.0,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,NaN,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Organizar Campos

In [ ]:
#@title Eliminar campos Name, Model y URLs por no ser útiles

ndf = ndf.drop(columns=["Name", "Model", "URL_data", "URL_image"])

ndf.head()

,Ship Type,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,ION LAUNCHER,GENERAL PURPOSE LAUNCHER,TURBOLASER TURRETS,BLASTER CANNONS,LASER TURRETS,TRACTOR BEAM PROJECTORS,TURBOLASER BATTERIES,ION CANNONS,TORPEDO LAUNCHER,GRAVITY WELL PROJECTORS,SUPERLASERS,BLASTER TURRETS,TORPEDO,ORBITAL MINE,THERMAL DETONATOR LAUNCHER,TURBOLASER,ION TURRETS,SPACE BOMB,ION BATTERIES,HULL-CUTTING AIRLOCKS,EMP CANNONS,TORPEDO TUBES
0,Landing Craft,3.0,40.0,4,15000.0,1 week,-1.0,none (Mark 2 has x10),35.0,45.0,1.0,-1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE Fighters,18.0,7.8,1,150.0,5 days,-1.0,NaN,110.0,14.0,0.0,24.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Medium Ships,8.0,298.0,150,100000000.0,6 months,2.0,x15,18.0,228.0,1.0,320.0,300,0.0,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Other Starfighters,8.0,7.4,1,55.0,1 day,-1.0,NaN,115.0,16.0,0.0,-1.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Bombers,18.0,6.3,1,65.0,2 days,-1.0,NaN,75.0,9.0,0.0,-1.0,NaN,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#@title Eliminar campos Crew, Consumables, Hyperdrive Backup & Troops no detectables por piloto

ndf = ndf.drop(columns=["Crew", "Consumables", "Hyperdrive Backup", "Troops" ])

ndf.head()

,Ship Type,Manufacturer,Length,Cargo Capacity,Hyperdrive Multiplier,Speed,Hull,Special Features,Shields,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,ION LAUNCHER,GENERAL PURPOSE LAUNCHER,TURBOLASER TURRETS,BLASTER CANNONS,LASER TURRETS,TRACTOR BEAM PROJECTORS,TURBOLASER BATTERIES,ION CANNONS,TORPEDO LAUNCHER,GRAVITY WELL PROJECTORS,SUPERLASERS,BLASTER TURRETS,TORPEDO,ORBITAL MINE,THERMAL DETONATOR LAUNCHER,TURBOLASER,ION TURRETS,SPACE BOMB,ION BATTERIES,HULL-CUTTING AIRLOCKS,EMP CANNONS,TORPEDO TUBES
0,Landing Craft,3.0,40.0,15000.0,-1.0,35.0,45.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE Fighters,18.0,7.8,150.0,-1.0,110.0,14.0,0.0,24.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Medium Ships,8.0,298.0,100000000.0,2.0,18.0,228.0,1.0,320.0,0.0,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Other Starfighters,8.0,7.4,55.0,-1.0,115.0,16.0,0.0,-1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Bombers,18.0,6.3,65.0,-1.0,75.0,9.0,0.0,-1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#@title Reordena columnas para que Ship Type quede al final

atributo_clase = 'Ship Type'

ndf = ndf.reindex(columns=(list([a for a in ndf.columns if a != atributo_clase] + [atributo_clase]) ))

ndf.head()

,Manufacturer,Length,Cargo Capacity,Hyperdrive Multiplier,Speed,Hull,Special Features,Shields,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,ION LAUNCHER,GENERAL PURPOSE LAUNCHER,TURBOLASER TURRETS,BLASTER CANNONS,LASER TURRETS,TRACTOR BEAM PROJECTORS,TURBOLASER BATTERIES,ION CANNONS,TORPEDO LAUNCHER,GRAVITY WELL PROJECTORS,SUPERLASERS,BLASTER TURRETS,TORPEDO,ORBITAL MINE,THERMAL DETONATOR LAUNCHER,TURBOLASER,ION TURRETS,SPACE BOMB,ION BATTERIES,HULL-CUTTING AIRLOCKS,EMP CANNONS,TORPEDO TUBES,Ship Type
0,3.0,40.0,15000.0,-1.0,35.0,45.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Landing Craft
1,18.0,7.8,150.0,-1.0,110.0,14.0,0.0,24.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TIE Fighters
2,8.0,298.0,100000000.0,2.0,18.0,228.0,1.0,320.0,0.0,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Medium Ships
3,8.0,7.4,55.0,-1.0,115.0,16.0,0.0,-1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Other Starfighters
4,18.0,6.3,65.0,-1.0,75.0,9.0,0.0,-1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TIE Bombers


#Controles Finales

In [ ]:
#@title Controlar ejemplos duplicados
detectar_ejemplos_duplicados = True #param {type:"boolean"}

if detectar_ejemplos_duplicados:

  # carga las columnas numéricas y no numéricas
  todasCols = list(ndf.columns)

  # combo selección de columnas
  selColumnas = widgets.SelectMultiple(
      options=todasCols,
      value=list(todasCols),
      description='Atributos seleccionados:',
      rows=len(todasCols),
      layout=Layout(display="flex")
  )

  ui = widgets.HBox([selColumnas])

  dups = None

  # Botones de acciones
  def mostrarDups(b):
    with output:
      global dups, ndf
      dupsMostrar = ndf.duplicated(keep=False)
      if dupsMostrar is not None and len(ndf[dupsMostrar])>0:
        display( ndf[dupsMostrar].sort_values(by=list(ndf.columns)) )
      else:
        print("\n No hay duplicados para mostrar!")
  def mostrarEstDups(b):
    with output:
      global dups, ndf
      dupsMostrar = ndf.duplicated(keep=False)
      if dupsMostrar is not None and len(ndf[dupsMostrar])>0:
        generar_estadisticas_detalladas(ndf[dupsMostrar], "Estadística de ejemplos duplicados")
      else:
        print("\n No hay duplicados para mostrar!")
  def eliminarDups(b):
    with output:
      clear_output()
      global dups, ndf
      if dups is not None and len(ndf[dups])>0:
        ndf = ndf[~dups] # deja una copia y borra el resto
        cambiaSeleccionDup(selColumnas.value)
        print("\n Ejemplos duplicados eliminados.")
      else:
        print("\n No hay duplicados para eliminar!")


  layoutButton = widgets.Layout(width='auto', height='40px') #set width and height
  buttonMD = widgets.Button(description="Mostrar Datos Duplicados",
                            display='flex',
                            flex_flow='column',
                            align_items='stretch',
                            layout = layoutButton)
  buttonMD.on_click(mostrarDups)
  buttonMeD = widgets.Button(description="Mostrar Estadísticas Duplicados",
                            display='flex',
                            flex_flow='column',
                            align_items='stretch',
                            layout = layoutButton)
  buttonMeD.on_click(mostrarEstDups)
  buttonED = widgets.Button(description="Eliminar Duplicados",
                            display='flex',
                            flex_flow='column',
                            align_items='stretch',
                            layout = layoutButton)
  buttonED.on_click(eliminarDups)
  output = widgets.Output()


  def cambiaSeleccionDup(cols):
    global dups
    # convierte la selección a un array
    arCols = np.array(cols)
    print("\n> Resultados de controlar ejemplos duplicados con atributos seleccionados: ")

    # detecta los ejemplos duplicados
    # (marca todos con 'True')
    dups = ndf.duplicated(subset=arCols, keep="first")
    # calcula las cantidades
    cantDup = len(ndf[dups])
    cantNoDup = len(ndf[~dups])

    # muestra resultados
    print("   - Cantidad de ejemplos con valores duplicados = ", cantDup, "(", round((cantDup/(cantDup+cantNoDup))*100,1), "%)")
    print("   + Cantidad de ejemplos con valores diferentes = ", cantNoDup, "(", round((cantNoDup/(cantDup+cantNoDup))*100,1), "%)")
    print("   = Cantidad total de ejemplos = ", (cantDup+cantNoDup))

    if cantDup > 0:
      # si hay duplicados
      print("\n> Posibles acciones a realizar:")
      display(buttonMD, buttonMeD, buttonED, output)

  out = widgets.interactive_output(cambiaSeleccionDup, {'cols': selColumnas})
  display(ui, out)

else:
  print("No se controlan datos duplicados.")


Output()

In [ ]:
#@title Controlar ejemplos inconsistentes  { run: "auto" }
#@markdown Nota: los ejemplos inconsistentes son los que tienen mismo valores de entrada pero distintos valores de salida

ejIncs = None

# Botones de acciones
def mostrarEstIncs(b):
  with outputInc:
    global ejIncs, ndf
    if ejIncs is not None and len(ndf[ejIncs])>0:
      generar_estadisticas_detalladas(ndf[ejIncs], "Estadística de ejemplos inconsistentes")
    else:
      print("\n No hay ejemplos inconsistentes para mostrar!")

def eliminarIncs(b):
  with outputInc:
    global ejIncs, ndf
    if ejIncs is not None and len(ndf[ejIncs])>0:
      ndf = ndf[~ejIncs] ## borra todos los inconsistentes!!!
      print("\n Ejemplos inconsistentes eliminados.")
      ejIncs = controlarInconsistencias()
    else:
      print("\n No hay ejemplos inconsistentes para eliminar!")


def controlarInconsistencias():
  if len(ndf)>0:
    print("\n> Resultados de controlar ejemplos inconsistentes: ")

    # determina columnas de entrada solamente
    arCols = list(ndf.columns)
    arCols.remove(atributo_clase)

    # detecta los ejemplos inconsistentes
    ###  https://stackoverflow.com/questions/61909261/how-can-we-detect-inconsistency-in-pandas-dataframe
    mask1 = ndf.duplicated(arCols, keep=False)
    mask2 = ~ndf.duplicated(arCols + [atributo_clase], keep=False)
    ejIncs = (mask1 & mask2)

    # calcula las cantidades
    cantInc = len(ndf[ejIncs])
    canNotInc = len(ndf[~ejIncs])

    # muestra resultados
    if cantInc > 0:
      # muestra datos ordenads
      print("\n Ejemplos inconsistentes detectados:")
      display(ndf[ejIncs].sort_values(by=list(arCols)))

  print("- Cantidad de ejemplos inconsistentes = ", cantInc, "(", round((cantInc/(cantInc+canNotInc))*100,1), "%)")
  print("+ Cantidad de ejemplos correctos = ", canNotInc, "(", round((canNotInc/(cantInc+canNotInc))*100,1), "%)")
  print("= Cantidad total de ejemplos = ", (cantInc+canNotInc))

  if cantInc > 0:
    # si hay duplicados
    print("\n> Posibles acciones a realizar:")
    display(buttonMeInc, buttonElimInc, outputInc)
  return ejIncs

# genera botones
layoutButton = widgets.Layout(width='auto', height='40px') #set width and height
buttonMeInc = widgets.Button(description="Mostrar Estadísticas Inconsistentes",
                          display='flex',
                          flex_flow='column',
                          align_items='stretch',
                          layout = layoutButton)
buttonMeInc.on_click(mostrarEstIncs)
buttonElimInc = widgets.Button(description="Eliminar Inconsistentes",
                          display='flex',
                          flex_flow='column',
                          align_items='stretch',
                          layout = layoutButton)
buttonElimInc.on_click(eliminarIncs)
outputInc = widgets.Output()

ejIncs = controlarInconsistencias()



> Resultados de controlar ejemplos inconsistentes: 
- Cantidad de ejemplos inconsistentes =  0 ( 0.0 %)
+ Cantidad de ejemplos correctos =  108 ( 100.0 %)
= Cantidad total de ejemplos =  108


# Estadísticas de Datos Preparados

In [ ]:
#@title Mostrar Estadísticas Generales

generar_estadisticas_detalladas(ndf, "> Estadísticas de los datos preparados")



 > Estadísticas de los datos preparados : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 0.00 ; 29.00 ],14.528 ± 7.244,1,0
Length,float64,[ 2.10 ; 16000.00 ],547.058 ± 2298.613,0,0
Cargo Capacity,float64,[ -1.00 ; 600000000.00 ],15073638.806 ± 75426742.857,0,0
Hyperdrive Multiplier,float64,[ -1.00 ; 4.00 ],0.747 ± 1.451,0,0
Speed,float64,[ 4.00 ; 155.00 ],69.120 ± 38.645,0,0
Hull,float64,[ 5.00 ; 77710.00 ],1658.759 ± 9115.872,0,0
Special Features,float64,[ 0.00 ; 1.00 ],0.204 ± 0.405,86,0
Shields,float64,[ -1.00 ; 136000.00 ],3107.352 ± 16578.845,0,0
Onboard Craft,float64,[ 0.00 ; 1.00 ],0.204 ± 0.405,86,0
LASER CANNONS,float64,[ 0.00 ; 5000.00 ],97.954 ± 676.613,39,0


Tiene 34 atributos y 108 ejemplos.




In [ ]:
#@title Mostrar Estadísticas por Clase { run: "auto" }


CLASES = list(ndf[atributo_clase].unique())
CLASES.sort()

for cl in CLASES:
  print("\n")
  auxDF = ndf.copy()
  auxDF = auxDF[ auxDF[atributo_clase] == cl]
  auxDF = auxDF.drop(labels=atributo_clase, axis=1)
  generar_estadisticas_detalladas(auxDF, "> Estadísticas de la clase "+ cl)






 > Estadísticas de la clase Command Ships : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 8.00 ; 10.00 ],8.667 ± 1.155,0,0
Length,float64,[ 10050.00 ; 16000.00 ],13683.333 ± 3186.037,0,0
Cargo Capacity,float64,[ 314844000.00 ; 600000000.00 ],438281333.333 ± 146381629.467,0,0
Hyperdrive Multiplier,float64,[ 2.00 ; 3.00 ],2.333 ± 0.577,0,0
Speed,float64,[ 6.00 ; 45.00 ],19.000 ± 22.517,0,0
Hull,float64,[ 24276.00 ; 77710.00 ],50756.333 ± 26720.145,0,0
Special Features,float64,[ 0.00 ; 1.00 ],0.667 ± 0.577,1,0
Shields,float64,[ 51000.00 ; 136000.00 ],94333.333 ± 42524.503,0,0
Onboard Craft,float64,[ 1.00 ; 1.00 ],1.000 ± 0.000,0,0
LASER CANNONS,float64,[ 0.00 ; 5000.00 ],3333.333 ± 2886.751,1,0


Tiene 33 atributos y 3 ejemplos.





 > Estadísticas de la clase Heavy Ships : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 8.00 ; 27.00 ],14.000 ± 5.684,0,0
Length,float64,[ 501.00 ; 1600.00 ],945.286 ± 407.904,0,0
Cargo Capacity,float64,[ -1.00 ; 36000000.00 ],11053571.357 ± 11989842.956,0,0
Hyperdrive Multiplier,float64,[ 0.60 ; 3.00 ],1.471 ± 0.682,0,0
Speed,float64,[ 4.00 ; 16.00 ],9.357 ± 2.872,0,0
Hull,float64,[ 760.00 ; 2272.00 ],1368.429 ± 545.535,0,0
Special Features,float64,[ 0.00 ; 1.00 ],0.214 ± 0.426,11,0
Shields,float64,[ 995.00 ; 5760.00 ],2786.786 ± 1317.853,0,0
Onboard Craft,float64,[ 0.00 ; 1.00 ],0.929 ± 0.267,1,0
LASER CANNONS,float64,[ 0.00 ; 80.00 ],16.857 ± 24.964,8,0


Tiene 33 atributos y 14 ejemplos.





 > Estadísticas de la clase Landing Craft : 


...

Tiene 33 atributos y 20 ejemplos.





 > Estadísticas de la clase TIE Support Craft : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 14.00 ; 18.00 ],17.600 ± 1.265,0,0
Length,float64,[ 6.30 ; 24.00 ],8.670 ± 5.438,0,0
Cargo Capacity,float64,[ 45.00 ; 150000.00 ],19780.500 ± 46340.032,0,0
Hyperdrive Multiplier,float64,[ -1.00 ; 2.00 ],-0.700 ± 0.949,0,0
Speed,float64,[ 70.00 ; 110.00 ],87.000 ± 12.517,0,0
Hull,float64,[ 9.00 ; 42.00 ],20.600 ± 12.920,0,0
Special Features,float64,[ 0.00 ; 0.00 ],0.000 ± 0.000,10,0
Shields,float64,[ -1.00 ; 20.00 ],3.200 ± 8.854,0,0
Onboard Craft,float64,[ 0.00 ; 0.00 ],0.000 ± 0.000,10,0
LASER CANNONS,float64,[ 0.00 ; 2.00 ],1.000 ± 0.471,1,0


Tiene 33 atributos y 10 ejemplos.





 > Estadísticas de la clase Transporters : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 28.00 ; 29.00 ],28.500 ± 0.577,0,0
Length,float64,[ 18.00 ; 44.00 ],31.250 ± 14.175,0,0
Cargo Capacity,float64,[ 100000.00 ; 900000.00 ],375000.000 ± 377491.722,0,0
Hyperdrive Multiplier,float64,[ 2.00 ; 2.00 ],2.000 ± 0.000,0,0
Speed,float64,[ 55.00 ; 67.00 ],59.500 ± 5.447,0,0
Hull,float64,[ 40.00 ; 142.00 ],98.250 ± 48.044,0,0
Special Features,float64,[ 0.00 ; 0.00 ],0.000 ± 0.000,4,0
Shields,float64,[ 80.00 ; 250.00 ],161.000 ± 77.914,0,0
Onboard Craft,float64,[ 0.00 ; 0.00 ],0.000 ± 0.000,4,0
LASER CANNONS,float64,[ 0.00 ; 8.00 ],2.500 ± 3.786,2,0


Tiene 33 atributos y 4 ejemplos.




#Exportar datos de Naves Imperiales

In [ ]:
#@title Exporta los datos como CSV

download_CSV = False #@param{type:"boolean"}

import os

archivo_datos_exportar = 'naves.csv'  #@param {type:"string"}

# crea directorio si no existe
if not os.path.isdir(path):
  os.mkdir(path)

def data2CSV(df, nomArch, descripcion):
  if df is None:
    print("No hay " + descripcion + " para exportar")
  df.to_csv(nomArch, index=False)
  if download_CSV:
    files.download(nomArch)
  print(descripcion + " exportados como ", nomArch)

# exporta datos
data2CSV(ndf, path + archivo_datos_exportar, "Datos preparados")


Datos preparados exportados como  /content/gdrive/MyDrive/demosColab/demoStarWars/datos/naves.csv
